# Notebook 2. The topology of the coordinative structure (Fig. 2)

Reproduces Fig. 2 and SI Table S3 with scripts 01 to 03 of the first paper, which this notebook holds (copies in the folder `src/paper1`). Reads no data. Run time about 6 minutes. `QUICK = True` in the first code cell shortens the runs for a test.

In [1]:
QUICK = False

import contextlib
import io
import types
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import networkx as nx

plt.rcParams.update({'figure.dpi': 110, 'font.size': 8, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.linewidth': 0.7,
                     'xtick.major.width': 0.7, 'ytick.major.width': 0.7,
                     'xtick.labelsize': 6.5, 'ytick.labelsize': 6.5, 'axes.labelsize': 7,
                     'legend.frameon': False, 'font.family': 'DejaVu Sans'})

P1_TOPOS = ['ER', 'WS', 'BA']                    # the three structures of the first paper
TOPOS = ['LAT', 'WS', 'ER', 'BA']                # from regular to scale-free, the order of the second paper
NAME = {'LAT': 'regular', 'WS': 'small-world', 'ER': 'random', 'BA': 'scale-free'}
TCOL = {'LAT': '#50C878', 'WS': '#9C27B0', 'ER': '#4A90E2', 'BA': '#E63946'}   # as in Figs. 3 and S4
HUB, PERI, MID = '#A4161A', '#2E5090', '#B8BEC6'                                # hub and periphery, as in Fig. 1C
GREY, INK = '#6E6E6E', '#222222'

# values published in the first paper: Table 3, S1 Appendix Table D, S2 Appendix Table B, Section 3.3
PUB = dict(kappa={'ER': (6.84, 0.38), 'WS': (6.09, 0.02), 'BA': (9.67, 0.59)},
           gamma=(2.78, 0.15), ks_p=(0.33, 0.18),
           dhp={'ER': (2.25, 0.37), 'WS': (1.46, 0.18), 'BA': (3.60, 0.62)},
           ar={'ER': (1.14, 0.09), 'WS': (1.02, 0.05), 'BA': (2.92, 0.31)},
           fr_targ={'ER': 0.33, 'WS': 0.29, 'BA': 0.84})
N_REAL = 20 if QUICK else 100

def published(key, t, fmt='.2f'):
    v = PUB[key].get(t)
    return 'not in the first paper' if v is None else format(v[0] if isinstance(v, tuple) else v, fmt)

## Step 0. The code of the first paper

In [2]:
# Script 01 of the first paper, simulations/01_network_generation.py, as released.
SRC_01 = r'''#!/usr/bin/env python3
"""
01_network_generation.py
========================
Coordinative Structures as Scale-Free Networks (Park, 2026)
Manuscript Section 2.1 — Network Topology

Generates three canonical network topologies with matched connectivity:
  - Erdős–Rényi (ER) random network
  - Watts–Strogatz (WS) small-world network
  - Barabási–Albert (BA) scale-free network

Computes structural metrics for manuscript Tables S1a–S1d and Table 2:
  - Degree statistics: ⟨k⟩, σ_k, k_max
  - Degree heterogeneity: κ = ⟨k²⟩/⟨k⟩
  - Power-law exponent: γ̂ (MLE) with KS goodness-of-fit
  - Centrality: eigenvector centrality, Gini coefficient, hub fraction
  - Topology: clustering coefficient, average path length

Parameters (from manuscript §2.1):
  N = 100 nodes
  ⟨k⟩ ≈ 6 (matched across topologies)
  ER: p = 0.0606
  WS: K = 6, β = 0.1
  BA: m = 3 (m₀ = 3 initial complete graph)
  Realizations: 100
  Base random seed: 42

Output:
  data/simulation_outputs/network_structural_metrics.csv
  data/simulation_outputs/network_ensemble_data.npz

Usage:
  python simulations/01_network_generation.py
"""

import numpy as np
import networkx as nx
from scipy import stats
import os
import sys
import time

# ══════════════════════════════════════════════════════════════
# PARAMETERS — manuscript §2.1, Tables S1b, S1d
# ══════════════════════════════════════════════════════════════
N = 100                   # Number of nodes (degrees of freedom)
N_REALIZATIONS = 100      # Independent network instances
BASE_SEED = 42            # For reproducibility

# Topology-specific parameters (Table S1b)
ER_P = 6.0 / (N - 1)     # ≈ 0.0606, targeting ⟨k⟩ ≈ 6
WS_K = 6                  # Ring neighbors (each side K/2 = 3)
WS_BETA = 0.1             # Rewiring probability
BA_M = 3                  # Edges per new node (⟨k⟩ → 2m = 6)

# Hub identification threshold
HUB_CRITERION = 'mean_plus_sd'  # C_e > μ + σ

# Output directory
OUTPUT_DIR = os.path.join(os.path.dirname(os.path.dirname(
    os.path.abspath(__file__))), 'data', 'simulation_outputs')


# ══════════════════════════════════════════════════════════════
# NETWORK GENERATION FUNCTIONS
# ══════════════════════════════════════════════════════════════
def generate_er(n, p, seed):
    """Erdős–Rényi G(N,p) random network (Eq S1 Appendix, Algorithm 1)."""
    return nx.erdos_renyi_graph(n, p, seed=seed)


def generate_ws(n, k, beta, seed):
    """Watts–Strogatz small-world network (Algorithm 2)."""
    return nx.watts_strogatz_graph(n, k, beta, seed=seed)


def generate_ba(n, m, seed):
    """Barabási–Albert scale-free network via preferential attachment (Eq 1)."""
    return nx.barabasi_albert_graph(n, m, seed=seed)


# ══════════════════════════════════════════════════════════════
# STRUCTURAL ANALYSIS FUNCTIONS
# ══════════════════════════════════════════════════════════════
def compute_degree_stats(G):
    """Compute degree distribution statistics."""
    degrees = np.array([G.degree(n) for n in G.nodes()])
    k_mean = degrees.mean()
    k_std = degrees.std()
    k_max = degrees.max()
    k_second_moment = (degrees ** 2).mean()
    kappa = k_second_moment / k_mean if k_mean > 0 else 0.0
    return {
        'k_mean': k_mean,
        'k_std': k_std,
        'k_max': k_max,
        'kappa': kappa,
        'degrees': degrees,
    }


def compute_centrality_stats(G):
    """Compute eigenvector centrality and derived measures."""
    try:
        ce = nx.eigenvector_centrality_numpy(G)
    except nx.NetworkXError:
        # Fallback for disconnected graphs
        ce = nx.eigenvector_centrality(G, max_iter=1000, tol=1e-6)
    ce_vals = np.array([ce[n] for n in G.nodes()])

    # Gini coefficient of eigenvector centrality
    gini = _gini_coefficient(ce_vals)

    # Hub fraction: nodes with C_e > μ + σ
    ce_mean = ce_vals.mean()
    ce_std = ce_vals.std()
    hub_fraction = np.mean(ce_vals > (ce_mean + ce_std))

    return {
        'gini_ce': gini,
        'hub_fraction': hub_fraction,
        'ce_values': ce_vals,
    }


def _gini_coefficient(values):
    """Compute Gini coefficient for a set of values."""
    sorted_vals = np.sort(values)
    n = len(sorted_vals)
    if n == 0 or sorted_vals.sum() == 0:
        return 0.0
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * sorted_vals) / (n * np.sum(sorted_vals))) - (n + 1) / n


def compute_topology_stats(G):
    """Compute clustering coefficient and average path length."""
    clustering = nx.average_clustering(G)

    # Average shortest path length (handle disconnected graphs)
    if nx.is_connected(G):
        avg_path = nx.average_shortest_path_length(G)
    else:
        # Use largest connected component
        gcc = max(nx.connected_components(G), key=len)
        subG = G.subgraph(gcc).copy()
        avg_path = nx.average_shortest_path_length(subG) if len(subG) > 1 else float('inf')

    return {
        'clustering': clustering,
        'avg_path_length': avg_path,
    }


def powerlaw_fit(degrees, k_min=None):
    """
    Maximum Likelihood Estimation of power-law exponent γ̂.

    Uses the discrete MLE formula (Eq S1 in S1 Appendix):
        γ̂ = 1 + n [Σᵢ ln(kᵢ / (k_min − 0.5))]⁻¹

    Following Clauset, Shalizi & Newman (2009) [Ref 58 in manuscript].

    Returns:
        gamma_hat: MLE exponent estimate
        ks_pvalue: KS test p-value against power-law distribution
        k_min_used: k_min threshold used
    """
    degrees = np.array(degrees)
    degrees = degrees[degrees > 0]  # Remove zeros

    if k_min is None:
        # Find optimal k_min by minimizing KS statistic
        unique_k = np.unique(degrees)
        unique_k = unique_k[unique_k >= 2]  # Minimum meaningful k_min
        best_ks = np.inf
        best_kmin = 2
        best_gamma = 3.0

        for km in unique_k:
            if km >= degrees.max():
                break
            tail = degrees[degrees >= km]
            if len(tail) < 10:
                break
            g = 1 + len(tail) * (np.sum(np.log(tail / (km - 0.5)))) ** (-1)
            if g > 1.5 and g < 5.0:  # Reasonable range
                # KS statistic
                cdf_emp = np.sort(tail)
                cdf_emp = np.arange(1, len(cdf_emp) + 1) / len(cdf_emp)
                cdf_theo = 1 - (np.sort(tail) / km) ** (-(g - 1))
                ks = np.max(np.abs(cdf_emp - cdf_theo))
                if ks < best_ks:
                    best_ks = ks
                    best_kmin = km
                    best_gamma = g

        k_min = best_kmin

    # Final fit with chosen k_min
    tail = degrees[degrees >= k_min]
    if len(tail) < 5:
        return np.nan, np.nan, k_min

    gamma_hat = 1 + len(tail) * (np.sum(np.log(tail / (k_min - 0.5)))) ** (-1)

    # KS goodness-of-fit via Monte Carlo (simplified)
    n_mc = 500
    ks_orig = _ks_stat_powerlaw(tail, gamma_hat, k_min)
    n_exceed = 0
    rng = np.random.RandomState(BASE_SEED)
    for _ in range(n_mc):
        # Generate synthetic power-law sample
        synth = _generate_powerlaw_sample(len(tail), gamma_hat, k_min, rng)
        if len(synth) < 5:
            continue
        g_synth = 1 + len(synth) * (np.sum(np.log(synth / (k_min - 0.5)))) ** (-1)
        ks_synth = _ks_stat_powerlaw(synth, g_synth, k_min)
        if ks_synth >= ks_orig:
            n_exceed += 1

    ks_pvalue = n_exceed / n_mc

    return gamma_hat, ks_pvalue, k_min


def _ks_stat_powerlaw(data, gamma, k_min):
    """KS statistic between data and power-law CDF."""
    sorted_data = np.sort(data)
    n = len(sorted_data)
    cdf_emp = np.arange(1, n + 1) / n
    cdf_theo = 1 - (sorted_data / k_min) ** (-(gamma - 1))
    return np.max(np.abs(cdf_emp - cdf_theo))


def _generate_powerlaw_sample(n, gamma, k_min, rng):
    """Generate discrete power-law random sample via inverse transform."""
    u = rng.uniform(0, 1, size=n)
    samples = np.floor(k_min * (1 - u) ** (-1.0 / (gamma - 1))).astype(int)
    return samples[samples >= k_min]


# ══════════════════════════════════════════════════════════════
# MAIN ENSEMBLE SIMULATION
# ══════════════════════════════════════════════════════════════
def run_ensemble():
    """
    Generate network ensembles and compute all structural metrics.
    Reproduces Tables S1b, S1d, and Table 2 (structural rows).
    """
    print("=" * 70)
    print("01_network_generation.py")
    print("Coordinative Structures as Scale-Free Networks")
    print(f"N = {N}, ⟨k⟩ ≈ 6, {N_REALIZATIONS} realizations, seed = {BASE_SEED}")
    print("=" * 70)

    topologies = {
        'ER': {'generator': lambda seed: generate_er(N, ER_P, seed),
               'params': f'p = {ER_P:.4f}'},
        'WS': {'generator': lambda seed: generate_ws(N, WS_K, WS_BETA, seed),
               'params': f'K = {WS_K}, β = {WS_BETA}'},
        'BA': {'generator': lambda seed: generate_ba(N, BA_M, seed),
               'params': f'm = {BA_M}'},
    }

    # Storage for ensemble results
    results = {}
    ensemble_data = {}

    for topo_name, topo_info in topologies.items():
        print(f"\n--- {topo_name} ({topo_info['params']}) ---")
        t0 = time.time()

        # Per-realization storage
        k_means = []
        k_stds = []
        k_maxs = []
        kappas = []
        clusterings = []
        path_lengths = []
        gini_ces = []
        hub_fracs = []
        gamma_hats = []
        ks_pvals = []
        all_degrees = []

        for r in range(N_REALIZATIONS):
            seed = BASE_SEED + r
            G = topo_info['generator'](seed)

            # Degree statistics
            dstats = compute_degree_stats(G)
            k_means.append(dstats['k_mean'])
            k_stds.append(dstats['k_std'])
            k_maxs.append(dstats['k_max'])
            kappas.append(dstats['kappa'])
            all_degrees.append(dstats['degrees'])

            # Centrality statistics
            cstats = compute_centrality_stats(G)
            gini_ces.append(cstats['gini_ce'])
            hub_fracs.append(cstats['hub_fraction'])

            # Topology statistics
            tstats = compute_topology_stats(G)
            clusterings.append(tstats['clustering'])
            path_lengths.append(tstats['avg_path_length'])

            # Power-law fit (BA only — ER and WS are not power-law)
            if topo_name == 'BA':
                gamma, ks_p, _ = powerlaw_fit(dstats['degrees'])
                gamma_hats.append(gamma)
                ks_pvals.append(ks_p)

        elapsed = time.time() - t0
        print(f"  Completed {N_REALIZATIONS} realizations in {elapsed:.1f}s")

        # Ensemble statistics
        res = {
            'k_mean': (np.mean(k_means), np.std(k_means)),
            'k_std': (np.mean(k_stds), np.std(k_stds)),
            'k_max': (np.mean(k_maxs), np.std(k_maxs)),
            'kappa': (np.mean(kappas), np.std(kappas)),
            'clustering': (np.mean(clusterings), np.std(clusterings)),
            'avg_path': (np.mean(path_lengths), np.std(path_lengths)),
            'gini_ce': (np.mean(gini_ces), np.std(gini_ces)),
            'hub_fraction': (np.mean(hub_fracs), np.std(hub_fracs)),
        }

        if topo_name == 'BA':
            valid_gamma = [g for g in gamma_hats if not np.isnan(g)]
            valid_ks = [p for p in ks_pvals if not np.isnan(p)]
            res['gamma_hat'] = (np.mean(valid_gamma), np.std(valid_gamma))
            res['ks_pvalue'] = (np.mean(valid_ks), np.std(valid_ks))

        results[topo_name] = res

        # Store ensemble arrays for downstream use
        ensemble_data[f'{topo_name}_kappas'] = np.array(kappas)
        ensemble_data[f'{topo_name}_gini'] = np.array(gini_ces)
        ensemble_data[f'{topo_name}_hub_frac'] = np.array(hub_fracs)
        ensemble_data[f'{topo_name}_k_means'] = np.array(k_means)
        ensemble_data[f'{topo_name}_k_stds'] = np.array(k_stds)
        ensemble_data[f'{topo_name}_k_maxs'] = np.array(k_maxs)
        ensemble_data[f'{topo_name}_clusterings'] = np.array(clusterings)
        ensemble_data[f'{topo_name}_path_lengths'] = np.array(path_lengths)

        # Store exemplar network (seed=42) for visualization
        G_exemplar = topo_info['generator'](BASE_SEED)
        pos_exemplar = nx.spring_layout(G_exemplar, seed=BASE_SEED,
                                        k=1.8 / np.sqrt(N), iterations=80)
        deg_exemplar = np.array([G_exemplar.degree(n) for n in G_exemplar.nodes()])
        try:
            ce_exemplar = np.array(list(
                nx.eigenvector_centrality_numpy(G_exemplar).values()))
        except Exception:
            ce_exemplar = np.zeros(N)

        ce_norm = (ce_exemplar - ce_exemplar.min()) / (
            ce_exemplar.max() - ce_exemplar.min() + 1e-10)

        pos_array = np.array([pos_exemplar[n] for n in G_exemplar.nodes()])
        edges_array = np.array(list(G_exemplar.edges()))

        ensemble_data[f'net_{topo_name}_pos'] = pos_array
        ensemble_data[f'net_{topo_name}_degs'] = deg_exemplar
        ensemble_data[f'net_{topo_name}_ce'] = ce_norm
        ensemble_data[f'net_{topo_name}_edges'] = edges_array

    return results, ensemble_data


def print_results(results):
    """Print results in Table S1d format."""
    print("\n" + "=" * 70)
    print("TABLE S1d — Structural validation measures")
    print(f"(ensemble means ± SD; {N_REALIZATIONS} realizations; "
          f"N = {N}, ⟨k⟩ ≈ 6, seed base = {BASE_SEED})")
    print("=" * 70)

    measures = [
        ('⟨k⟩', 'k_mean'),
        ('σ_k', 'k_std'),
        ('k_max', 'k_max'),
        ('κ = ⟨k²⟩/⟨k⟩', 'kappa'),
        ('Clustering', 'clustering'),
        ('Avg path', 'avg_path'),
        ('Gini(C_e)', 'gini_ce'),
        ('Hub fraction', 'hub_fraction'),
    ]

    header = f"{'Measure':<18} {'ER (Random)':<20} {'WS (Small-World)':<20} {'BA (Scale-Free)':<20}"
    print(header)
    print("-" * 78)

    for label, key in measures:
        row = f"{label:<18}"
        for topo in ['ER', 'WS', 'BA']:
            mean, sd = results[topo][key]
            row += f" {mean:>7.2f} ± {sd:<6.2f}   "
        print(row)

    # BA-specific measures
    if 'gamma_hat' in results['BA']:
        mean, sd = results['BA']['gamma_hat']
        print(f"{'γ̂ (MLE)':<18} {'---':<20} {'---':<20} {mean:>7.2f} ± {sd:<6.2f}")
    if 'ks_pvalue' in results['BA']:
        mean, sd = results['BA']['ks_pvalue']
        print(f"{'KS p-value':<18} {'---':<20} {'---':<20} {mean:>7.2f} ± {sd:<6.2f}")

    # Manuscript Table 2 comparison
    print("\n" + "=" * 70)
    print("VERIFICATION against manuscript Table 2")
    print("=" * 70)
    expected = {
        'ER': {'kappa': 6.84, 'gini_ce': 0.26, 'hub_fraction': 0.16},
        'WS': {'kappa': 6.09, 'gini_ce': 0.16, 'hub_fraction': 0.16},
        'BA': {'kappa': 9.67, 'gini_ce': 0.35, 'hub_fraction': 0.10},
    }
    for topo in ['ER', 'WS', 'BA']:
        print(f"\n  {topo}:")
        for key, exp_val in expected[topo].items():
            sim_val = results[topo][key][0]
            match = "✓" if abs(sim_val - exp_val) / (exp_val + 1e-10) < 0.15 else "✗"
            print(f"    {key:<15} expected={exp_val:.2f}  simulated={sim_val:.2f}  {match}")


def save_results(results, ensemble_data):
    """Save results to CSV and NPZ files."""
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # CSV summary
    csv_path = os.path.join(OUTPUT_DIR, 'network_structural_metrics.csv')
    with open(csv_path, 'w') as f:
        f.write("topology,measure,mean,sd\n")
        for topo in ['ER', 'WS', 'BA']:
            for key, (mean, sd) in results[topo].items():
                f.write(f"{topo},{key},{mean:.6f},{sd:.6f}\n")
    print(f"\nSaved: {csv_path}")

    # NPZ ensemble data (for downstream scripts and figure generation)
    npz_path = os.path.join(OUTPUT_DIR, 'network_ensemble_data.npz')
    np.savez_compressed(npz_path, **ensemble_data)
    print(f"Saved: {npz_path}")


# ══════════════════════════════════════════════════════════════
# ENTRY POINT
# ══════════════════════════════════════════════════════════════
if __name__ == '__main__':
    results, ensemble_data = run_ensemble()
    print_results(results)
    save_results(results, ensemble_data)
    print("\n✓ 01_network_generation.py complete.")
'''

In [3]:
# Script 02 of the first paper, simulations/02_cascade_dynamics.py, as released.
SRC_02 = r'''#!/usr/bin/env python3
"""
02_cascade_dynamics.py
======================
Manuscript Section 2.2 — Cascade Dynamics (Eqs 2–4)

Simulates DOF recruitment cascades on three network topologies,
implementing the six-step update cycle from S2 Appendix:
  1. Spontaneous potential generation (probability p_n per node)
  2. Neighbor-mediated propagation (probability p_l per active link)
  3. Stability evaluation (Eq 2)
  4. State transition (Eq 3)
  5. Coordination strength update
  6. Cascade termination check

Computes cascade measures for Tables S3a, S3b, and Table 2:
  - R_early = |A(τ=5)| / |A(0)|  (early propagation rate)
  - R_∞ = |A(t→∞)| / N  (cascade reach)
  - Δ_hp = R_early(hub) / R_early(peripheral)  (asymmetry ratio)

Parameters (Table S2):
  p_n = 0.1       Spontaneous activation probability
  p_l = 0.1       Link propagation probability
  p_max = 0.99    Maximum stability
  f_0 = 0.7       Baseline stability
  f_1 = 0.7       Centrality weight
  c_p = 1.0       Reference constant
  τ = 5           Early-stage window

For topology-isolated trials (Table S3b): p_n = 0, single seed.

Output:
  data/simulation_outputs/cascade_dynamics_results.csv
  data/simulation_outputs/cascade_ensemble_data.npz

Usage:
  python simulations/02_cascade_dynamics.py
"""

import numpy as np
import networkx as nx
import os
import sys
import time

# ══════════════════════════════════════════════════════════════
# PARAMETERS — Table S2 (S2 Appendix)
# ══════════════════════════════════════════════════════════════
# Network parameters (from 01_network_generation.py)
N = 100
BASE_SEED = 42
N_REALIZATIONS = 100
N_CASCADE_TRIALS = 100    # Topology-isolated trials per realization

# Cascade model parameters (Table S2)
P_N = 0.1          # Spontaneous activation probability
P_L = 0.1          # Link propagation probability
P_MAX = 0.99       # Maximum stability (protection probability)
F_0 = 0.7          # Baseline stability floor
F_1 = 0.7          # Eigenvector centrality contribution
C_P = 1.0          # Stability function scaling constant
TAU = 5            # Early-stage window for R_early
T_MAX = 50         # Maximum cascade timesteps
F_M = 0.0          # Memory decay factor (implicit in strength reset)

# Network generation parameters
ER_P = 6.0 / (N - 1)
WS_K = 6
WS_BETA = 0.1
BA_M = 3

# Hub/peripheral definitions
HUB_THRESHOLD_SIGMA = 2   # Hub: degree > ⟨k⟩ + 2σ
# Peripheral: degree < ⟨k⟩

OUTPUT_DIR = os.path.join(os.path.dirname(os.path.dirname(
    os.path.abspath(__file__))), 'data', 'simulation_outputs')


# ══════════════════════════════════════════════════════════════
# NETWORK GENERATORS
# ══════════════════════════════════════════════════════════════
def generate_network(topo, seed):
    if topo == 'ER':
        return nx.erdos_renyi_graph(N, ER_P, seed=seed)
    elif topo == 'WS':
        return nx.watts_strogatz_graph(N, WS_K, WS_BETA, seed=seed)
    elif topo == 'BA':
        return nx.barabasi_albert_graph(N, BA_M, seed=seed)


# ══════════════════════════════════════════════════════════════
# CASCADE DYNAMICS MODEL — Eqs 2–4
# ══════════════════════════════════════════════════════════════
def stability_function(f_pi, s):
    """
    Eq 2: π_i = p_max / [1 + c_p / (f_π_i × s_i)]

    Returns protection probability for a DOF.
    When f_pi * s → 0:  π → 0 (no stability)
    When f_pi * s → ∞:  π → p_max
    At f_pi * s = c_p:  π = p_max / 2
    """
    denom = f_pi * s
    if denom < 1e-12:
        return 0.0
    return P_MAX / (1.0 + C_P / denom)


def compute_f_pi(centrality_i):
    """Eq 4: f_π_i = f_0 + f_1 × C_i"""
    return F_0 + F_1 * centrality_i


def run_single_cascade(G, ce_dict, seed_node, rng, use_spontaneous=False):
    """
    Run a single cascade from a seed node through the six-step update.

    For topology-isolated trials (Table S3b): use_spontaneous=False, p_n=0.

    Returns:
        trajectory: list of |A(t)| at each timestep
    """
    n_nodes = G.number_of_nodes()
    nodes = list(G.nodes())

    # State variables per node
    potential = np.zeros(n_nodes, dtype=int)     # φ_i(t) ∈ {0,1}
    active = np.zeros(n_nodes, dtype=int)        # A_i(t) ∈ {0,1}
    strength = np.ones(n_nodes) * 0.1            # s_i(t) ≥ 0

    # Precompute centrality-based stability fraction (Eq 4)
    f_pi = np.array([compute_f_pi(ce_dict.get(n, 0.0)) for n in nodes])

    # Adjacency list for efficiency
    adj = {n: list(G.neighbors(n)) for n in nodes}

    # Initialize: seed node has potential
    seed_idx = nodes.index(seed_node)
    potential[seed_idx] = 1

    trajectory = [1]  # |A(0)| = 1 (seed has potential)

    for t in range(1, T_MAX + 1):
        new_potential = potential.copy()
        new_active = active.copy()

        # Step 1: Spontaneous activation (probability p_n per node)
        if use_spontaneous:
            for i in range(n_nodes):
                if not active[i] and not potential[i]:
                    if rng.random() < P_N:
                        new_potential[i] = 1

        # Step 2: Neighbor-mediated propagation (p_l per link)
        for i in range(n_nodes):
            if active[i]:
                for j_node in adj[nodes[i]]:
                    j = nodes.index(j_node) if j_node not in nodes[:n_nodes] else j_node
                    # Use node index directly
                    j_idx = list(G.nodes()).index(j_node)
                    if not active[j_idx] and not new_potential[j_idx]:
                        if rng.random() < P_L:
                            new_potential[j_idx] = 1

        # Step 3–4: Stability evaluation (Eq 2) and state transition (Eq 3)
        for i in range(n_nodes):
            if new_potential[i] and not active[i]:
                pi_i = stability_function(f_pi[i], strength[i])
                # Eq 3: P(A_i = 1 | φ_i > 0) = 1 − π_i
                if rng.random() < (1.0 - pi_i):
                    new_active[i] = 1
                    new_potential[i] = 0
                    strength[i] = 0.0  # Reset upon activation

        # Step 5: Coordination strength update (for non-active nodes)
        for i in range(n_nodes):
            if not new_active[i]:
                strength[i] = 1.0 + (1.0 - F_M - f_pi[i]) * strength[i]

        potential = new_potential
        active = new_active
        trajectory.append(int(active.sum()))

        # Step 6: Cascade termination check
        if active.sum() == n_nodes:
            # Pad remaining timesteps
            for _ in range(t + 1, T_MAX + 1):
                trajectory.append(n_nodes)
            break
        if potential.sum() == 0 and (active.sum() == trajectory[-2] if len(trajectory) > 1 else True):
            # No more potential to spread — check if cascade stalled
            any_can_spread = False
            for i in range(n_nodes):
                if active[i]:
                    for j_node in adj[nodes[i]]:
                        j_idx = list(G.nodes()).index(j_node)
                        if not active[j_idx]:
                            any_can_spread = True
                            break
                if any_can_spread:
                    break
            if not any_can_spread and potential.sum() == 0:
                for _ in range(t + 1, T_MAX + 1):
                    trajectory.append(int(active.sum()))
                break

    return trajectory


def run_single_cascade_fast(G, ce_array, seed_idx, rng):
    """
    Optimized cascade for topology-isolated trials (p_n = 0).
    Uses numpy indexing for speed.
    """
    n = G.number_of_nodes()
    nodes = sorted(G.nodes())
    node_to_idx = {n: i for i, n in enumerate(nodes)}

    # Build adjacency as index lists
    adj_idx = [[] for _ in range(n)]
    for u, v in G.edges():
        adj_idx[node_to_idx[u]].append(node_to_idx[v])
        adj_idx[node_to_idx[v]].append(node_to_idx[u])

    # Stability fractions
    f_pi = F_0 + F_1 * ce_array

    # State
    active = np.zeros(n, dtype=bool)
    potential = np.zeros(n, dtype=bool)
    strength = np.ones(n) * 0.1

    potential[seed_idx] = True
    trajectory = [1]

    for t in range(1, T_MAX + 1):
        new_potential = potential.copy()
        new_active = active.copy()

        # Step 2: Propagation from active nodes (no spontaneous: p_n = 0)
        active_indices = np.where(active)[0]
        for i in active_indices:
            for j in adj_idx[i]:
                if not active[j] and not new_potential[j]:
                    if rng.random() < P_L:
                        new_potential[j] = True

        # Steps 3–4: State transition
        pot_indices = np.where(new_potential & ~active)[0]
        for i in pot_indices:
            pi_i = stability_function(f_pi[i], strength[i])
            if rng.random() < (1.0 - pi_i):
                new_active[i] = True
                new_potential[i] = False
                strength[i] = 0.0

        # Step 5: Strength accumulation
        inactive = ~new_active
        strength[inactive] = 1.0 + (1.0 - f_pi[inactive]) * strength[inactive]

        potential = new_potential
        active = new_active
        n_active = int(active.sum())
        trajectory.append(n_active)

        # Step 6: Termination
        if n_active == n:
            trajectory.extend([n] * (T_MAX - t))
            break

    return trajectory


# ══════════════════════════════════════════════════════════════
# CASCADE MEASURES
# ══════════════════════════════════════════════════════════════
def compute_cascade_measures(trajectory, n_nodes):
    """Compute R_early and R_∞ from a trajectory."""
    # R_early = |A(τ)| / |A(0)|, with |A(0)| = 1
    r_early = trajectory[min(TAU, len(trajectory) - 1)]
    r_inf = trajectory[-1] / n_nodes
    return r_early, r_inf


def identify_hub_peripheral(G):
    """
    Identify hub and peripheral nodes.
    Hub: degree > ⟨k⟩ + 2σ
    Peripheral: degree < ⟨k⟩
    """
    degrees = np.array([G.degree(n) for n in sorted(G.nodes())])
    k_mean = degrees.mean()
    k_std = degrees.std()
    nodes = sorted(G.nodes())

    hubs = [nodes[i] for i in range(len(nodes))
            if degrees[i] > k_mean + HUB_THRESHOLD_SIGMA * k_std]
    peripherals = [nodes[i] for i in range(len(nodes))
                   if degrees[i] < k_mean]
    return hubs, peripherals


# ══════════════════════════════════════════════════════════════
# MAIN ENSEMBLE SIMULATION
# ══════════════════════════════════════════════════════════════
def run_ensemble():
    """
    Run topology-isolated cascade trials across all topologies.
    Reproduces Tables S3b and Table 2 (cascade rows).
    """
    print("=" * 70)
    print("02_cascade_dynamics.py")
    print("Cascade Dynamics Model (Eqs 2–4)")
    print(f"N = {N}, τ = {TAU}, p_l = {P_L}, {N_REALIZATIONS} realizations "
          f"× {N_CASCADE_TRIALS} trials")
    print("=" * 70)

    topologies = ['ER', 'WS', 'BA']
    results = {}
    ensemble_data = {}

    for topo in topologies:
        print(f"\n--- {topo} ---")
        t0 = time.time()

        all_r_early_hub = []
        all_r_early_periph = []
        all_r_inf_hub = []
        all_r_inf_periph = []
        all_delta_hp = []

        for r in range(N_REALIZATIONS):
            seed = BASE_SEED + r
            G = generate_network(topo, seed)
            nodes = sorted(G.nodes())
            node_to_idx = {n: i for i, n in enumerate(nodes)}

            # Eigenvector centrality
            try:
                ce = nx.eigenvector_centrality_numpy(G)
            except Exception:
                ce = {n: 1.0 / N for n in G.nodes()}
            ce_array = np.array([ce[n] for n in nodes])

            hubs, peripherals = identify_hub_peripheral(G)

            if not hubs:
                # Fallback: use top 5% by degree
                degrees = np.array([G.degree(n) for n in nodes])
                top_k = max(1, N // 20)
                hub_indices = np.argsort(degrees)[-top_k:]
                hubs = [nodes[i] for i in hub_indices]

            if not peripherals:
                degrees = np.array([G.degree(n) for n in nodes])
                bot_k = max(1, N // 5)
                periph_indices = np.argsort(degrees)[:bot_k]
                peripherals = [nodes[i] for i in periph_indices]

            rng = np.random.RandomState(seed)

            # Hub-initiated cascades
            r_early_hub_trials = []
            r_inf_hub_trials = []
            for trial in range(min(N_CASCADE_TRIALS, len(hubs) * 20)):
                hub_node = hubs[trial % len(hubs)]
                hub_idx = node_to_idx[hub_node]
                traj = run_single_cascade_fast(G, ce_array, hub_idx, rng)
                re, ri = compute_cascade_measures(traj, N)
                r_early_hub_trials.append(re)
                r_inf_hub_trials.append(ri)

            # Peripheral-initiated cascades
            r_early_periph_trials = []
            r_inf_periph_trials = []
            for trial in range(min(N_CASCADE_TRIALS, len(peripherals) * 5)):
                periph_node = peripherals[trial % len(peripherals)]
                periph_idx = node_to_idx[periph_node]
                traj = run_single_cascade_fast(G, ce_array, periph_idx, rng)
                re, ri = compute_cascade_measures(traj, N)
                r_early_periph_trials.append(re)
                r_inf_periph_trials.append(ri)

            mean_re_hub = np.mean(r_early_hub_trials)
            mean_re_periph = np.mean(r_early_periph_trials)
            mean_ri_hub = np.mean(r_inf_hub_trials)
            mean_ri_periph = np.mean(r_inf_periph_trials)

            delta_hp = mean_re_hub / max(mean_re_periph, 1e-10)

            all_r_early_hub.append(mean_re_hub)
            all_r_early_periph.append(mean_re_periph)
            all_r_inf_hub.append(mean_ri_hub)
            all_r_inf_periph.append(mean_ri_periph)
            all_delta_hp.append(delta_hp)

            if (r + 1) % 20 == 0:
                print(f"  Realization {r+1}/{N_REALIZATIONS}...")

        elapsed = time.time() - t0
        print(f"  Completed in {elapsed:.1f}s")

        results[topo] = {
            'R_early_hub': (np.mean(all_r_early_hub), np.std(all_r_early_hub)),
            'R_early_periph': (np.mean(all_r_early_periph), np.std(all_r_early_periph)),
            'R_inf_hub': (np.mean(all_r_inf_hub), np.std(all_r_inf_hub)),
            'R_inf_periph': (np.mean(all_r_inf_periph), np.std(all_r_inf_periph)),
            'Delta_hp': (np.mean(all_delta_hp), np.std(all_delta_hp)),
        }

        ensemble_data[f'{topo}_delta_hp'] = np.array(all_delta_hp)
        ensemble_data[f'{topo}_r_early_hub'] = np.array(all_r_early_hub)
        ensemble_data[f'{topo}_r_early_periph'] = np.array(all_r_early_periph)

    return results, ensemble_data


def print_results(results):
    """Print results matching Table S3b format."""
    print("\n" + "=" * 70)
    print("TABLE S3b — Cascade dynamics ensemble statistics")
    print(f"({N_REALIZATIONS} realizations × {N_CASCADE_TRIALS} trials; p_n = 0)")
    print("=" * 70)

    header = f"{'Measure':<20} {'ER':<20} {'WS':<20} {'BA':<20}"
    print(header)
    print("-" * 80)

    measures = [
        ('R_early (hub)', 'R_early_hub'),
        ('R_early (periph)', 'R_early_periph'),
        ('R_∞ (hub)', 'R_inf_hub'),
        ('R_∞ (periph)', 'R_inf_periph'),
        ('Δ_hp', 'Delta_hp'),
    ]

    for label, key in measures:
        row = f"{label:<20}"
        for topo in ['ER', 'WS', 'BA']:
            m, s = results[topo][key]
            row += f" {m:>7.2f} ± {s:<6.2f}   "
        print(row)

    # Verification against manuscript Table 2
    print("\n" + "=" * 70)
    print("VERIFICATION against manuscript Table 2 / Table S3b")
    print("=" * 70)
    expected_delta = {'ER': 2.25, 'WS': 1.46, 'BA': 3.60}
    expected_re_hub = {'ER': 7.58, 'WS': 4.98, 'BA': 14.15}
    for topo in ['ER', 'WS', 'BA']:
        dhp_sim = results[topo]['Delta_hp'][0]
        dhp_exp = expected_delta[topo]
        re_sim = results[topo]['R_early_hub'][0]
        re_exp = expected_re_hub[topo]
        print(f"  {topo}: Δ_hp expected={dhp_exp:.2f} simulated={dhp_sim:.2f} | "
              f"R_early(hub) expected={re_exp:.2f} simulated={re_sim:.2f}")


def save_results(results, ensemble_data):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    csv_path = os.path.join(OUTPUT_DIR, 'cascade_dynamics_results.csv')
    with open(csv_path, 'w') as f:
        f.write("topology,measure,mean,sd\n")
        for topo in ['ER', 'WS', 'BA']:
            for key, (mean, sd) in results[topo].items():
                f.write(f"{topo},{key},{mean:.6f},{sd:.6f}\n")
    print(f"\nSaved: {csv_path}")

    npz_path = os.path.join(OUTPUT_DIR, 'cascade_ensemble_data.npz')
    np.savez_compressed(npz_path, **ensemble_data)
    print(f"Saved: {npz_path}")


if __name__ == '__main__':
    results, ensemble_data = run_ensemble()
    print_results(results)
    save_results(results, ensemble_data)
    print("\n✓ 02_cascade_dynamics.py complete.")
'''

In [4]:
# Script 03 of the first paper, simulations/03_percolation_robustness.py, as released.
SRC_03 = r'''#!/usr/bin/env python3
"""
03_percolation_robustness.py
============================
Manuscript Sections 2.3 (Correspondences 1–3) & 3.3 — Percolation and Robustness

Implements two complementary analyses:
  A. Bond percolation sweeps (Eqs 6, 9, 10)
  B. Robustness-fragility analysis (Eq 7)

Part A — Percolation Dynamics:
  - Order parameter: P_∞(p) = |C₁(p)| / N  (Eq 9)
  - Susceptibility:  χ(p) = dP_∞ / dp  (Eq 10)
  - Percolation threshold: p_c = ⟨k⟩ / (⟨k²⟩ − ⟨k⟩) (Eq 6)
  Sweep p ∈ [0, 1] in steps of 0.01, 100 realizations.

Part B — Robustness-Fragility:
  - Fragmentation ratio: f_r(q) = 1 − S(q) / S(0)  (Eq 7)
  - Asymmetry ratio: A_r(q) = f_r(targeted, q) / f_r(random, q)
  Removal fractions q ∈ {0.01, ..., 0.50}, 50 random trials.

Target outputs: Tables S4, S5, S5b, and Table 2 (percolation rows).

Parameters (Table S6b):
  Bond occupation sweep: p ∈ [0, 1], step 0.01
  P_∞ threshold: 0.5
  Network realizations: 100

Output:
  data/simulation_outputs/percolation_results.csv
  data/simulation_outputs/robustness_results.csv
  data/simulation_outputs/percolation_ensemble_data.npz

Usage:
  python simulations/03_percolation_robustness.py
"""

import numpy as np
import networkx as nx
import os
import time

# ══════════════════════════════════════════════════════════════
# PARAMETERS
# ══════════════════════════════════════════════════════════════
N = 100
BASE_SEED = 42
N_REALIZATIONS = 100

# Network parameters
ER_P = 6.0 / (N - 1)
WS_K = 6
WS_BETA = 0.1
BA_M = 3

# Percolation sweep parameters (Table S6b)
P_RANGE = np.arange(0.0, 1.01, 0.01)    # Bond occupation p ∈ [0, 1]
P_INF_THRESHOLD = 0.5                     # Giant component threshold

# Robustness parameters (S3 Appendix Part C)
Q_VALUES = np.array([0.01, 0.02, 0.03, 0.04, 0.05,
                     0.10, 0.15, 0.20, 0.25, 0.28, 0.30, 0.40, 0.50])
N_RANDOM_TRIALS = 50   # Random removal trials per q value
Q_REPORT = 0.28        # Primary reporting threshold for A_r

OUTPUT_DIR = os.path.join(os.path.dirname(os.path.dirname(
    os.path.abspath(__file__))), 'data', 'simulation_outputs')


# ══════════════════════════════════════════════════════════════
# NETWORK GENERATORS
# ══════════════════════════════════════════════════════════════
def generate_network(topo, seed):
    if topo == 'ER':
        return nx.erdos_renyi_graph(N, ER_P, seed=seed)
    elif topo == 'WS':
        return nx.watts_strogatz_graph(N, WS_K, WS_BETA, seed=seed)
    elif topo == 'BA':
        return nx.barabasi_albert_graph(N, BA_M, seed=seed)


# ══════════════════════════════════════════════════════════════
# PART A: BOND PERCOLATION — Eqs 6, 9, 10
# ══════════════════════════════════════════════════════════════
def bond_percolation_sweep(G, p_range, rng):
    """
    Perform bond percolation sweep over the range of occupation probabilities.

    For each p, independently retain each edge with probability p,
    then compute the giant component fraction P_∞(p) = |C₁(p)| / N (Eq 9).

    Returns:
        p_inf: array of P_∞ values for each p
    """
    edges = list(G.edges())
    n_edges = len(edges)
    n_nodes = G.number_of_nodes()

    p_inf = np.zeros(len(p_range))

    for i, p in enumerate(p_range):
        if p == 0:
            p_inf[i] = 1.0 / n_nodes  # Each node is its own component
            continue
        if p >= 1.0:
            if nx.is_connected(G):
                p_inf[i] = 1.0
            else:
                gcc = max(nx.connected_components(G), key=len)
                p_inf[i] = len(gcc) / n_nodes
            continue

        # Retain each edge with probability p
        mask = rng.random(n_edges) < p
        retained = [edges[j] for j in range(n_edges) if mask[j]]

        H = nx.Graph()
        H.add_nodes_from(G.nodes())
        H.add_edges_from(retained)

        if H.number_of_edges() == 0:
            p_inf[i] = 1.0 / n_nodes
        else:
            gcc = max(nx.connected_components(H), key=len)
            p_inf[i] = len(gcc) / n_nodes

    return p_inf


def compute_susceptibility(p_range, p_inf):
    """
    Eq 10: χ(p) = dP_∞ / dp

    Compute numerical derivative using central differences.
    """
    chi = np.gradient(p_inf, p_range)
    return chi


def find_percolation_threshold(p_range, p_inf, threshold=0.5):
    """Find p_c where P_∞ first exceeds threshold."""
    above = np.where(p_inf >= threshold)[0]
    if len(above) == 0:
        return p_range[-1]
    return p_range[above[0]]


def theoretical_pc(G):
    """
    Eq 6: p_c = ⟨k⟩ / (⟨k²⟩ − ⟨k⟩)

    Molloy–Reed criterion for uncorrelated networks.
    """
    degrees = np.array([G.degree(n) for n in G.nodes()])
    k_mean = degrees.mean()
    k2_mean = (degrees ** 2).mean()
    kappa = k2_mean / k_mean
    if kappa <= 1:
        return 1.0
    return 1.0 / (kappa - 1)


# ══════════════════════════════════════════════════════════════
# PART B: ROBUSTNESS-FRAGILITY — Eq 7
# ══════════════════════════════════════════════════════════════
def largest_component_size(G):
    """Return size of the largest connected component."""
    if G.number_of_nodes() == 0:
        return 0
    return len(max(nx.connected_components(G), key=len))


def robustness_random_removal(G, q_values, n_trials, rng):
    """
    Random node removal: for each q, remove ⌊q·N⌋ nodes randomly,
    compute S(q), average over n_trials.

    Returns:
        f_r_random: fragmentation ratios f_r(q) = 1 − S(q)/S(0) (Eq 7)
    """
    s0 = largest_component_size(G)
    nodes = list(G.nodes())
    n_nodes = len(nodes)
    f_r = np.zeros(len(q_values))

    for i, q in enumerate(q_values):
        n_remove = max(1, int(q * n_nodes))
        s_trials = []
        for _ in range(n_trials):
            to_remove = rng.choice(nodes, size=n_remove, replace=False)
            H = G.copy()
            H.remove_nodes_from(to_remove)
            s_trials.append(largest_component_size(H))
        f_r[i] = 1.0 - np.mean(s_trials) / s0

    return f_r


def robustness_targeted_removal(G, q_values):
    """
    Targeted hub removal: remove nodes in decreasing degree order.

    Returns:
        f_r_targeted: fragmentation ratios f_r(q) = 1 − S(q)/S(0)
    """
    s0 = largest_component_size(G)
    nodes = sorted(G.nodes(), key=lambda n: G.degree(n), reverse=True)
    n_nodes = len(nodes)
    f_r = np.zeros(len(q_values))

    for i, q in enumerate(q_values):
        n_remove = max(1, int(q * n_nodes))
        H = G.copy()
        H.remove_nodes_from(nodes[:n_remove])
        s_q = largest_component_size(H)
        f_r[i] = 1.0 - s_q / s0

    return f_r


# ══════════════════════════════════════════════════════════════
# MAIN ENSEMBLE SIMULATION
# ══════════════════════════════════════════════════════════════
def run_ensemble():
    print("=" * 70)
    print("03_percolation_robustness.py")
    print("Percolation Dynamics (Eqs 6, 9, 10) & Robustness (Eq 7)")
    print(f"N = {N}, {N_REALIZATIONS} realizations, seed = {BASE_SEED}")
    print("=" * 70)

    topologies = ['ER', 'WS', 'BA']
    results = {}
    ensemble_data = {}

    for topo in topologies:
        print(f"\n--- {topo} ---")
        t0 = time.time()

        all_p_inf = np.zeros((N_REALIZATIONS, len(P_RANGE)))
        all_pc_sim = []
        all_pc_theo = []
        all_p_peak = []
        all_chi_max = []

        all_fr_random = np.zeros((N_REALIZATIONS, len(Q_VALUES)))
        all_fr_targeted = np.zeros((N_REALIZATIONS, len(Q_VALUES)))
        all_ar = []

        for r in range(N_REALIZATIONS):
            seed = BASE_SEED + r
            G = generate_network(topo, seed)
            rng = np.random.RandomState(seed)

            # Part A: Percolation sweep
            p_inf = bond_percolation_sweep(G, P_RANGE, rng)
            chi = compute_susceptibility(P_RANGE, p_inf)
            pc_sim = find_percolation_threshold(P_RANGE, p_inf)
            pc_theo = theoretical_pc(G)
            p_peak = P_RANGE[np.argmax(chi)]
            chi_max = np.max(chi)

            all_p_inf[r] = p_inf
            all_pc_sim.append(pc_sim)
            all_pc_theo.append(pc_theo)
            all_p_peak.append(p_peak)
            all_chi_max.append(chi_max)

            # Part B: Robustness
            fr_rand = robustness_random_removal(G, Q_VALUES, N_RANDOM_TRIALS, rng)
            fr_targ = robustness_targeted_removal(G, Q_VALUES)

            all_fr_random[r] = fr_rand
            all_fr_targeted[r] = fr_targ

            # A_r at q = 0.28
            q_idx = np.argmin(np.abs(Q_VALUES - Q_REPORT))
            ar = fr_targ[q_idx] / max(fr_rand[q_idx], 1e-10)
            all_ar.append(ar)

            if (r + 1) % 20 == 0:
                print(f"  Realization {r+1}/{N_REALIZATIONS}...")

        elapsed = time.time() - t0
        print(f"  Completed in {elapsed:.1f}s")

        # Compute transition width Δp (P_∞ from 0.1 to 0.9)
        mean_p_inf = all_p_inf.mean(axis=0)
        p01_idx = np.where(mean_p_inf >= 0.1)[0]
        p09_idx = np.where(mean_p_inf >= 0.9)[0]
        p01 = P_RANGE[p01_idx[0]] if len(p01_idx) > 0 else 0.0
        p09 = P_RANGE[p09_idx[0]] if len(p09_idx) > 0 else 1.0
        delta_p = p09 - p01

        q_idx = np.argmin(np.abs(Q_VALUES - Q_REPORT))

        results[topo] = {
            'pc_sim': (np.mean(all_pc_sim), np.std(all_pc_sim)),
            'pc_theo': (np.mean(all_pc_theo), np.std(all_pc_theo)),
            'p_peak': (np.mean(all_p_peak), np.std(all_p_peak)),
            'chi_max': (np.mean(all_chi_max), np.std(all_chi_max)),
            'delta_p': (delta_p, 0.0),
            'fr_random_028': (all_fr_random[:, q_idx].mean(),
                              all_fr_random[:, q_idx].std()),
            'fr_targeted_028': (all_fr_targeted[:, q_idx].mean(),
                                all_fr_targeted[:, q_idx].std()),
            'A_r': (np.mean(all_ar), np.std(all_ar)),
        }

        # Store curves for figure generation
        ensemble_data[f'{topo}_p_inf_mean'] = mean_p_inf
        ensemble_data[f'{topo}_p_inf_std'] = all_p_inf.std(axis=0)
        ensemble_data[f'{topo}_chi_mean'] = compute_susceptibility(P_RANGE, mean_p_inf)
        ensemble_data[f'{topo}_fr_random_mean'] = all_fr_random.mean(axis=0)
        ensemble_data[f'{topo}_fr_targeted_mean'] = all_fr_targeted.mean(axis=0)
        ensemble_data[f'{topo}_ar_values'] = np.array(all_ar)

    ensemble_data['p_range'] = P_RANGE
    ensemble_data['q_values'] = Q_VALUES
    return results, ensemble_data


def print_results(results):
    print("\n" + "=" * 70)
    print("TABLE S5b — Percolation dynamics and robustness-fragility")
    print(f"(N = {N}, ⟨k⟩ ≈ 6, {N_REALIZATIONS} realizations)")
    print("=" * 70)

    header = f"{'Measure':<22} {'ER':<20} {'WS':<20} {'BA':<20}"
    print(header)
    print("-" * 82)

    measures = [
        ('p_c (P_∞ ≥ 0.5)', 'pc_sim'),
        ('p_c (theoretical)', 'pc_theo'),
        ('p_peak', 'p_peak'),
        ('χ_max', 'chi_max'),
        ('Δp (0.1→0.9)', 'delta_p'),
        ('f_r(random, 0.28)', 'fr_random_028'),
        ('f_r(targeted, 0.28)', 'fr_targeted_028'),
        ('A_r(0.28)', 'A_r'),
    ]

    for label, key in measures:
        row = f"{label:<22}"
        for topo in ['ER', 'WS', 'BA']:
            m, s = results[topo][key]
            if s > 0:
                row += f" {m:>7.3f} ± {s:<6.3f}  "
            else:
                row += f" {m:>7.3f}            "
        print(row)

    # Verification
    print("\n" + "=" * 70)
    print("VERIFICATION against manuscript Table 2 / Table S5b")
    print("=" * 70)
    expected = {
        'ER': {'p_peak': 0.22, 'A_r': 1.14},
        'WS': {'p_peak': 0.27, 'A_r': 1.02},
        'BA': {'p_peak': 0.16, 'A_r': 2.92},
    }
    for topo in ['ER', 'WS', 'BA']:
        pp_sim = results[topo]['p_peak'][0]
        ar_sim = results[topo]['A_r'][0]
        pp_exp = expected[topo]['p_peak']
        ar_exp = expected[topo]['A_r']
        print(f"  {topo}: p_peak expected={pp_exp:.2f} sim={pp_sim:.2f} | "
              f"A_r expected={ar_exp:.2f} sim={ar_sim:.2f}")


def save_results(results, ensemble_data):
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Percolation CSV
    csv_path = os.path.join(OUTPUT_DIR, 'percolation_results.csv')
    with open(csv_path, 'w') as f:
        f.write("topology,measure,mean,sd\n")
        for topo in ['ER', 'WS', 'BA']:
            for key, (mean, sd) in results[topo].items():
                f.write(f"{topo},{key},{mean:.6f},{sd:.6f}\n")
    print(f"\nSaved: {csv_path}")

    # Ensemble NPZ
    npz_path = os.path.join(OUTPUT_DIR, 'percolation_ensemble_data.npz')
    np.savez_compressed(npz_path, **ensemble_data)
    print(f"Saved: {npz_path}")


if __name__ == '__main__':
    results, ensemble_data = run_ensemble()
    print_results(results)
    save_results(results, ensemble_data)
    print("\n✓ 03_percolation_robustness.py complete.")
'''

In [5]:
def load_script(name, src, fixes=()):
    for old, new in fixes:
        assert src.count(old) == 1, old
        src = src.replace(old, new)
    m = types.ModuleType(name)
    m.__file__ = name + '.py'
    exec(compile(src, name + '.py', 'exec'), m.__dict__)
    m.N_REALIZATIONS = N_REAL
    return m

FIX_01 = [('    except nx.NetworkXError:\n        # Fallback for disconnected graphs',
           '    except nx.NetworkXException:\n        # Fallback for disconnected graphs')]
P01 = load_script('p1_01_network_generation', SRC_01, FIX_01)
P02 = load_script('p1_02_cascade_dynamics', SRC_02)
P03 = load_script('p1_03_percolation_robustness', SRC_03)
WS_BETA = P01.WS_BETA

def lattice_pass(module):
    # run the script's own ensemble once more with the rewiring probability at zero, output kept quiet
    module.WS_BETA = 0.0
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            return module.run_ensemble()
    finally:
        module.WS_BETA = WS_BETA

print('NetworkX', nx.__version__, '| NumPy', np.__version__)
print('scripts loaded; realizations per topology:', N_REAL, '| base seed:', P01.BASE_SEED,
      '| small-world rewiring probability:', WS_BETA)

NetworkX 3.7 | NumPy 2.5.3
scripts loaded; realizations per topology: 100 | base seed: 42 | small-world rewiring probability: 0.1


## Step 1. The four structures and their degree distributions (panels A to D and F)

In [6]:
RES01, ENS01 = P01.run_ensemble()
P01.print_results(RES01)
RES01_LAT, _ = lattice_pass(P01)
same = all(np.isclose(RES01_LAT[t][k][0], RES01[t][k][0]) for t in ['ER', 'BA'] for k in RES01[t])
print()
print('lattice pass of script 01; check, its random and scale-free rows repeat the first pass:', same)
RES01['LAT'] = RES01_LAT['WS']

gen = {'LAT': lambda s: P01.generate_ws(P01.N, P01.WS_K, 0.0, s),
       'WS': lambda s: P01.generate_ws(P01.N, P01.WS_K, WS_BETA, s),
       'ER': lambda s: P01.generate_er(P01.N, P01.ER_P, s),
       'BA': lambda s: P01.generate_ba(P01.N, P01.BA_M, s)}
GAM6, KSP6, DEGS = [], [], {t: [] for t in TOPOS}
for r in range(N_REAL):
    for t in TOPOS:
        d = P01.compute_degree_stats(gen[t](P01.BASE_SEED + r))['degrees']
        DEGS[t].append(d)
        if t == 'BA':
            g, p, _ = P01.powerlaw_fit(d, k_min=6)
            GAM6.append(g); KSP6.append(p)
DEGS = {t: np.concatenate(v) for t, v in DEGS.items()}
GAM6, KSP6 = np.array(GAM6), np.array(KSP6)

KAPPA = {t: (RES01[t]['kappa'][0], RES01[t]['kappa'][1]) for t in TOPOS}
FEAT = {t: dict(clus=RES01[t]['clustering'][0], kmax=RES01[t]['k_max'][0]) for t in TOPOS}
print('degree heterogeneity kappa, mean and sd over', N_REAL, 'realizations: '
      + ', '.join(f'{NAME[t]} {KAPPA[t][0]:.2f} ± {KAPPA[t][1]:.2f}' for t in TOPOS)
      + ' | published: ' + ', '.join(f'{NAME[t]} {published("kappa", t)}' for t in TOPOS))
print('clustering and largest degree, mean over realizations: '
      + ', '.join(f'{NAME[t]} {FEAT[t]["clus"]:.2f} and {FEAT[t]["kmax"]:.1f}' for t in TOPOS))
print(f'power-law exponent, scale-free, k_min = 6: {GAM6.mean():.2f} ± {GAM6.std():.2f}, '
      f'KS p {KSP6.mean():.2f} ± {KSP6.std():.2f} | published {PUB["gamma"][0]:.2f} ± {PUB["gamma"][1]:.2f}, '
      f'KS p {PUB["ks_p"][0]:.2f} ± {PUB["ks_p"][1]:.2f}')
print(f'power-law exponent, scale-free, k_min chosen by the KS statistic as in script 01: '
      f'{RES01["BA"]["gamma_hat"][0]:.2f} ± {RES01["BA"]["gamma_hat"][1]:.2f}')

NET = {}
for t in TOPOS:
    G = gen[t](P01.BASE_SEED)
    hubs, peri = P02.identify_hub_peripheral(G)
    NET[t] = dict(G=G, pos=ENS01.get(f'net_{t}_pos'), deg=np.array([G.degree(i) for i in sorted(G)]),
                  hubs=set(hubs), peri=set(peri))
    print(f'seed-42 {NAME[t]} structure: {G.number_of_edges()} couplings, largest degree '
          f'{NET[t]["deg"].max()}, {len(hubs)} hubs, {len(peri)} peripheral elements')

01_network_generation.py
Coordinative Structures as Scale-Free Networks
N = 100, ⟨k⟩ ≈ 6, 100 realizations, seed = 42

--- ER (p = 0.0606) ---
  Completed 100 realizations in 0.9s

--- WS (K = 6, β = 0.1) ---
  Completed 100 realizations in 0.9s

--- BA (m = 3) ---
  Completed 100 realizations in 2.4s

TABLE S1d — Structural validation measures
(ensemble means ± SD; 100 realizations; N = 100, ⟨k⟩ ≈ 6, seed base = 42)
Measure            ER (Random)          WS (Small-World)     BA (Scale-Free)     
------------------------------------------------------------------------------
⟨k⟩                   5.94 ± 0.33         6.00 ± 0.00         5.82 ± 0.00     
σ_k                   2.31 ± 0.18         0.75 ± 0.09         4.72 ± 0.36     
k_max                12.42 ± 1.42         8.18 ± 0.54        29.29 ± 4.60     
κ = ⟨k²⟩/⟨k⟩          6.84 ± 0.38         6.09 ± 0.02         9.67 ± 0.59     
Clustering            0.06 ± 0.01         0.45 ± 0.03         0.15 ± 0.03     
Avg path              2

## Step 2. The hub-first cascade (panel E)

In [7]:
LOG, NOW, ALIAS = [], {}, {'ER': 'ER', 'WS': 'WS', 'BA': 'BA'}
_cascade, _generate = P02.run_single_cascade_fast, P02.generate_network

def seeds_counted_as_hubs(G):
    # the hubs run_ensemble of script 02 seeds from, with its fallback when there are none
    hubs, _ = P02.identify_hub_peripheral(G)
    if not hubs:
        nodes = sorted(G.nodes()); degrees = np.array([G.degree(n) for n in nodes])
        hubs = [nodes[i] for i in np.argsort(degrees)[-max(1, P02.N // 20):]]
    return set(hubs)

def generate_logged(topo, seed):
    G = _generate(topo, seed)
    NOW.update(topo=ALIAS[topo], seed=seed, hubs=seeds_counted_as_hubs(G))
    return G

def cascade_logged(G, ce_array, seed_idx, rng):
    traj = _cascade(G, ce_array, seed_idx, rng)
    if NOW['topo'] is not None:
        LOG.append((NOW['topo'], NOW['seed'], seed_idx in NOW['hubs'], np.array(traj)))
    return traj

P02.generate_network, P02.run_single_cascade_fast = generate_logged, cascade_logged
RES02, ENS02 = P02.run_ensemble()
P02.print_results(RES02)
ALIAS.update(ER=None, WS='LAT', BA=None)
RES02_LAT, ENS02_LAT = lattice_pass(P02)
P02.generate_network, P02.run_single_cascade_fast = _generate, _cascade
same_pass = all(np.allclose(ENS02_LAT[f'{t}_delta_hp'], ENS02[f'{t}_delta_hp']) for t in ['ER', 'BA'])
ENS02['LAT_delta_hp'] = ENS02_LAT['WS_delta_hp']

TRAJ, DHP = {}, {}
for t in TOPOS:
    seeds_t = sorted({s for (tt, s, h, tr) in LOG if tt == t})
    hub = np.array([np.mean([tr for (tt, s, h, tr) in LOG if tt == t and s == sd and h], axis=0) for sd in seeds_t])
    per = np.array([np.mean([tr for (tt, s, h, tr) in LOG if tt == t and s == sd and not h], axis=0) for sd in seeds_t])
    TRAJ[t] = dict(hub=hub, per=per)
    DHP[t] = hub[:, P02.TAU] / per[:, P02.TAU]
RATIO = {t: TRAJ[t]['hub'] / TRAJ[t]['per'] for t in TOPOS}     # hub over peripheral, per realization and step
same = all(np.allclose(DHP[t], ENS02[f'{t}_delta_hp']) for t in TOPOS)
print()
print('lattice pass of script 02; check, its random and scale-free rows repeat the first pass:', same_pass)
print('check, the asymmetry recomputed from the recorded trajectories equals the script\'s own:', same)
print('cascade asymmetry Delta_hp at tau = 5, mean and sd: '
      + ', '.join(f'{NAME[t]} {DHP[t].mean():.2f} ± {DHP[t].std():.2f}' for t in TOPOS)
      + ' | published: ' + ', '.join(f'{NAME[t]} {published("dhp", t)}' for t in TOPOS))
print('elements active at tau = 5 after a hub seed, mean: '
      + ', '.join(f'{NAME[t]} {TRAJ[t]["hub"][:, P02.TAU].mean():.2f}' for t in TOPOS)
      + ' | after a peripheral seed: '
      + ', '.join(f'{NAME[t]} {TRAJ[t]["per"][:, P02.TAU].mean():.2f}' for t in TOPOS))
print(f'scale-free realizations with Delta_hp above 2: {(DHP["BA"] > 2).sum()} of {len(DHP["BA"])}, '
      f'range {DHP["BA"].min():.2f} to {DHP["BA"].max():.2f}')

02_cascade_dynamics.py
Cascade Dynamics Model (Eqs 2–4)
N = 100, τ = 5, p_l = 0.1, 100 realizations × 100 trials

--- ER ---
  Realization 20/100...
  Realization 40/100...
  Realization 60/100...
  Realization 80/100...
  Realization 100/100...
  Completed in 37.5s

--- WS ---
  Realization 20/100...
  Realization 40/100...
  Realization 60/100...
  Realization 80/100...
  Realization 100/100...
  Completed in 32.2s

--- BA ---
  Realization 20/100...
  Realization 40/100...
  Realization 60/100...
  Realization 80/100...
  Realization 100/100...
  Completed in 35.4s

TABLE S3b — Cascade dynamics ensemble statistics
(100 realizations × 100 trials; p_n = 0)
Measure              ER                   WS                   BA                  
--------------------------------------------------------------------------------
R_early (hub)           5.75 ± 0.65         4.02 ± 0.35        10.19 ± 1.21     
R_early (periph)        2.74 ± 0.26         2.84 ± 0.20         2.98 ± 0.26     
R_∞ (hu

## Step 3. Robust yet fragile (panel G)

In [8]:
ROB = {t: dict(rand=[], targ=[]) for t in TOPOS}
_rand, _targ, _gen3 = P03.robustness_random_removal, P03.robustness_targeted_removal, P03.generate_network
ALIAS.update(ER='ER', WS='WS', BA='BA')

def gen3_logged(topo, seed):
    NOW.update(topo=ALIAS[topo], seed=seed)
    return _gen3(topo, seed)

def rand_logged(G, q_values, n_trials, rng):
    fr = _rand(G, q_values, n_trials, rng)
    if NOW['topo'] is not None:
        ROB[NOW['topo']]['rand'].append(fr)
    return fr

def targ_logged(G, q_values):
    fr = _targ(G, q_values)
    if NOW['topo'] is not None:
        ROB[NOW['topo']]['targ'].append(fr)
    return fr

P03.generate_network, P03.robustness_random_removal, P03.robustness_targeted_removal = gen3_logged, rand_logged, targ_logged
RES03, ENS03 = P03.run_ensemble()
P03.print_results(RES03)
ALIAS.update(ER=None, WS='LAT', BA=None)
RES03_LAT, ENS03_LAT = lattice_pass(P03)
P03.generate_network, P03.robustness_random_removal, P03.robustness_targeted_removal = _gen3, _rand, _targ
same_pass = all(np.allclose(ENS03_LAT[f'{t}_ar_values'], ENS03[f'{t}_ar_values']) for t in ['ER', 'BA'])
ENS03['LAT_ar_values'] = ENS03_LAT['WS_ar_values']
ENS03['LAT_fr_random_mean'] = ENS03_LAT['WS_fr_random_mean']

Q = np.concatenate([[0.0], ENS03['q_values']])
QI = int(np.argmin(np.abs(ENS03['q_values'] - P03.Q_REPORT)))
FR = {t: {k: np.array(v) for k, v in ROB[t].items()} for t in TOPOS}
KEEP = {t: {k: np.hstack([np.ones((len(v), 1)), 1 - v]) for k, v in FR[t].items()} for t in TOPOS}
AR = {t: ENS03[f'{t}_ar_values'] for t in TOPOS}
same = all(np.allclose(FR[t]['targ'][:, QI] / np.maximum(FR[t]['rand'][:, QI], 1e-10), AR[t])
           and np.allclose(FR[t]['rand'].mean(0), ENS03[f'{t}_fr_random_mean']) for t in TOPOS)
print()
print('lattice pass of script 03; check, its random and scale-free rows repeat the first pass:', same_pass)
print('check, the asymmetry recomputed from the recorded removals equals the script\'s own:', same)
print('robustness asymmetry A_r at q = 0.28, mean and sd: '
      + ', '.join(f'{NAME[t]} {AR[t].mean():.2f} ± {AR[t].std():.2f}' for t in P1_TOPOS)
      + ' | published: ' + ', '.join(f'{NAME[t]} {published("ar", t)}' for t in P1_TOPOS))
print('fragmentation under targeted removal at q = 0.28, mean: '
      + ', '.join(f'{NAME[t]} {RES03[t]["fr_targeted_028"][0]:.2f}' for t in P1_TOPOS)
      + ' | published: ' + ', '.join(f'{NAME[t]} {published("fr_targ", t)}' for t in P1_TOPOS))
print(f'regular lattice, largest component left at q = 0.28 under random removal: '
      f'{KEEP["LAT"]["rand"][:, QI + 1].mean():.2f}, against '
      + ', '.join(f'{NAME[t]} {KEEP[t]["rand"][:, QI + 1].mean():.2f}' for t in P1_TOPOS))
print(f'scale-free realizations with A_r above 1: {(AR["BA"] > 1).sum()} of {len(AR["BA"])}, '
      f'range {AR["BA"].min():.2f} to {AR["BA"].max():.2f}')

03_percolation_robustness.py
Percolation Dynamics (Eqs 6, 9, 10) & Robustness (Eq 7)
N = 100, 100 realizations, seed = 42

--- ER ---
  Realization 20/100...
  Realization 40/100...
  Realization 60/100...
  Realization 80/100...
  Realization 100/100...
  Completed in 38.4s

--- WS ---
  Realization 20/100...
  Realization 40/100...
  Realization 60/100...
  Realization 80/100...
  Realization 100/100...
  Completed in 40.2s

--- BA ---
  Realization 20/100...
  Realization 40/100...
  Realization 60/100...
  Realization 80/100...
  Realization 100/100...
  Completed in 39.1s

TABLE S5b — Percolation dynamics and robustness-fragility
(N = 100, ⟨k⟩ ≈ 6, 100 realizations)
Measure                ER                   WS                   BA                  
----------------------------------------------------------------------------------
p_c (P_∞ ≥ 0.5)          0.224 ± 0.025      0.277 ± 0.026      0.206 ± 0.020   
p_c (theoretical)        0.172 ± 0.011      0.196 ± 0.001      0.116 ± 

## Step 4. The seven panels, one figure each

In [9]:
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.lines import Line2D

mpl.rcParams.update({'font.size': 7, 'axes.labelsize': 6.4, 'xtick.labelsize': 6.0, 'ytick.labelsize': 6.0,
                     'legend.fontsize': 6.0, 'axes.linewidth': 0.6, 'xtick.major.width': 0.6,
                     'ytick.major.width': 0.6, 'xtick.major.size': 2.4, 'ytick.major.size': 0})

FIG_W = 6.93                            # the width of the figures of the second paper, inches
PW_T, PH_T = FIG_W / 4, 1.90            # structure panel canvas
PW_B, PH_B = FIG_W / 3, 1.52            # plotting panel canvas
NET_H = 1.54                            # each structure fills a square of this side
BOX_L = (0.42, 0.33, 1.62, 0.94)        # axes box of a plotting panel, y axis on the left
BOX_R = (0.27, 0.33, 1.62, 0.94)        # the same box with the y axis on the right
MIRROR = {L: False for L in 'ABCDEFG'}  # every letter at the left, every y axis on the left

# the palette of the second paper
INK, NEUTRAL, LINK = '#1A1A1A', '#37474F', '#C3CBD0'
_HOT = plt.get_cmap('hot_r')(np.linspace(0.06, 0.94, 256))
_COOL = plt.get_cmap('Blues')(np.linspace(0.92, 0.06, 256))
CMAP = LinearSegmentedColormap.from_list('hot_div', np.vstack([_COOL, _HOT]))
VLIM = 2.7157337518864986               # the colour scale of the second paper, in bits
CNORM = Normalize(-VLIM, VLIM)
# the four structures in the palette of Fig. 1, with the hues Figs. 3 and S4 give them
TONE = {'LAT': '#2E7D4E', 'WS': '#7A5195', 'ER': '#2E5090', 'BA': '#A4161A'}
MARK = {'LAT': 'o', 'WS': 's', 'ER': 'D', 'BA': '^'}
LABEL = {'LAT': 'Regular lattice', 'WS': 'Small-world', 'ER': 'Random', 'BA': 'Scale-free'}
GENLINE = {'LAT': r'ring,  $\beta$ = 0', 'WS': r'rewired,  $\beta$ = 0.1',
           'ER': r'Erdős–Rényi,  $p$ = 0.061', 'BA': r'preferential attachment,  $m$ = 3'}

def giant(G):
    return G if nx.is_connected(G) else G.subgraph(max(nx.connected_components(G), key=len)).copy()

def eig_shares(G):
    c = nx.eigenvector_centrality_numpy(G)
    v = np.abs(np.array([c[n] for n in G.nodes()], dtype=float))
    return np.round(v / v.sum(), 12)

def spread(G, w, iterations=900, tie=None):
    # the spring rule of the second paper: seeded on a golden-angle spiral by rank in share, then relaxed;
    # tie, if given, orders elements of equal share and degree in place of their labels
    idx = {n: i for i, n in enumerate(G.nodes())}
    tie = tie or {z: z for z in G.nodes()}
    order = sorted(G.nodes(), key=lambda z: (-round(float(w[idx[z]]), 9), -G.degree(z), tie[z]))
    n = len(order)
    seed_pos = {node: np.array([np.sqrt(t / max(n - 1, 1)) * np.cos(t * np.pi * (3 - np.sqrt(5))),
                                np.sqrt(t / max(n - 1, 1)) * np.sin(t * np.pi * (3 - np.sqrt(5)))])
                for t, node in enumerate(order)}
    pos = nx.spring_layout(G, pos=seed_pos, seed=3, iterations=iterations, k=2.8 / np.sqrt(n))
    r = np.array([np.hypot(*pos[z]) for z in G.nodes()])
    s = np.sqrt(np.sqrt((r ** 2).mean()) * r.max()) or 1.0
    return {z: (v[0] / s, v[1] / s) for z, v in pos.items()}

def colour(w, n):
    b = np.log2(np.asarray(w, float) * n)
    return CMAP(CNORM(np.where(np.abs(b) < 1e-9, 0.0, b)))

SHOW = {t: giant(gen[t](P01.BASE_SEED)) for t in TOPOS}
SHOW_W = {t: eig_shares(SHOW[t]) for t in TOPOS}
# Every element of the lattice has the same share and degree, so the spiral would seed them in label
# order and the relaxed spring would draw a circle. Its ties are broken in a fixed random order and
# the relaxation stops after 12 steps, so that it is drawn as a network like the other three.
LAT_TIE = dict(zip(SHOW['LAT'].nodes(), np.random.default_rng(3).permutation(SHOW['LAT'].number_of_nodes())))
POS = {t: spread(SHOW[t], SHOW_W[t], iterations=12, tie=LAT_TIE) if t == 'LAT' else spread(SHOW[t], SHOW_W[t])
       for t in TOPOS}
# each structure fills its own square, centred on its own extent
FRAME = {}
for t in TOPOS:
    xy_ = np.array(list(POS[t].values())); c_ = (xy_.max(0) + xy_.min(0)) / 2
    FRAME[t] = (c_, 0.5 * 1.08 * (xy_.max(0) - xy_.min(0)).max())
KMAX = float(max(max(dict(SHOW[t].degree()).values()) for t in TOPOS))

def axes_at(fig, ox, oy, box):
    W, H = fig.get_size_inches()
    l, b, w, h = box
    return fig.add_axes([(ox + l) / W, (oy + b) / H, w / W, h / H])

def heading(fig, ox, oy, pw, ph, letter, title, sub=None):
    # the panel letter at the outer edge in bold, the title centred, a line beneath it
    W, H = fig.get_size_inches(); y = (oy + ph - 0.19) / H
    right = MIRROR[letter]
    fig.text((ox + (pw - 0.05 if right else 0.05)) / W, y, letter, fontsize=9.4, fontweight='bold',
             color='k', ha='right' if right else 'left', va='bottom')
    fig.text((ox + pw / 2) / W, y, title, fontsize=7.9, color='k', ha='center', va='bottom')
    if sub:
        fig.text((ox + pw / 2) / W, y - 0.02 / H, sub, fontsize=6.2, color='k', ha='center', va='top')

def frame(ax, xt, yt, right=False, log=False):
    # a line above and below, none at the sides, dotted rules, ticks at the minimum, middle and maximum
    if log:
        ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlim(xt[0], xt[-1]); ax.set_ylim(yt[0], yt[-1])
    ax.set_xticks(xt); ax.set_yticks(yt)
    fmt = mpl.ticker.FuncFormatter(lambda v, _: f'{v:g}')
    ax.xaxis.set_major_formatter(fmt); ax.yaxis.set_major_formatter(fmt)
    ax.xaxis.set_minor_locator(mpl.ticker.NullLocator()); ax.yaxis.set_minor_locator(mpl.ticker.NullLocator())
    for s_ in ('left', 'right'):
        ax.spines[s_].set_visible(False)
    for s_ in ('top', 'bottom'):
        ax.spines[s_].set_visible(True); ax.spines[s_].set_linewidth(0.7)
    ax.grid(True, axis='y', alpha=0.40, linestyle=':', linewidth=0.5, color='gray')
    ax.set_axisbelow(True)
    if right:
        ax.yaxis.tick_right(); ax.yaxis.set_label_position('right')
    ax.tick_params(axis='y', length=0, pad=1.4)
    ax.tick_params(axis='x', length=2.4, width=0.6, direction='out', pad=1.4)

def guide(ax, x, text, ytext, ha='left'):
    ax.axvline(x, color=INK, lw=0.7, ls=(0, (3.2, 2.2)), zorder=1)
    ax.text(x, ytext, text, fontsize=6.0, color='k', ha=ha, va='bottom',
            transform=mpl.transforms.blended_transform_factory(ax.transData, ax.transAxes))

def draw_structure(fig, ox, oy, t, letter):
    W, H = fig.get_size_inches()
    ax = axes_at(fig, ox, oy, ((PW_T - NET_H) / 2, 0.02, NET_H, NET_H))
    (cx, cy), half = FRAME[t]
    ax.set_xlim(cx - half, cx + half); ax.set_ylim(cy - half, cy + half); ax.set_aspect('equal'); ax.axis('off')
    G, w, pos = SHOW[t], SHOW_W[t], POS[t]
    n_all = P01.N
    gap = lambda u, v: min(abs(u - v), n_all - abs(u - v))
    near = [(pos[u], pos[v]) for u, v in G.edges() if t != 'WS' or gap(u, v) <= P01.WS_K // 2]
    far = [(pos[u], pos[v]) for u, v in G.edges() if t == 'WS' and gap(u, v) > P01.WS_K // 2]
    ax.add_collection(LineCollection(near, colors=LINK, linewidths=0.30, alpha=0.75, zorder=1))
    if far:
        ax.add_collection(LineCollection(far, colors=NEUTRAL, linewidths=0.45, alpha=0.60, zorder=2))
    nodes = list(G.nodes()); d = np.array([G.degree(n) for n in nodes], dtype=float)
    xy = np.array([pos[n] for n in nodes]); order = np.argsort(d, kind='stable')
    ax.scatter(xy[order, 0], xy[order, 1], s=(1.6 + 46.0 * d / KMAX)[order],
               c=colour(w, len(nodes))[order], edgecolors=NEUTRAL, linewidths=0.25, zorder=3)
    heading(fig, ox, oy, PW_T, PH_T, letter, LABEL[t], GENLINE[t])

def draw_degree(fig, ox, oy, letter):
    ax = axes_at(fig, ox, oy, BOX_R if MIRROR[letter] else BOX_L)
    frame(ax, [1, 10, 100], [0.01, 0.1, 1], right=MIRROR[letter], log=True)
    for t in TOPOS:
        dd = DEGS[t]; ks = np.arange(1, dd.max() + 1)
        cc = np.array([(dd >= k).mean() for k in ks])
        ax.plot(ks, cc, '-', color=TONE[t], lw=0.9, zorder=3)
        if dd.min() == dd.max():                                    # every degree the same: one step
            ax.plot([dd.max()] * 2, [cc[-1], 1e-3], '-', color=TONE[t], lw=0.9, zorder=3)
        ax.plot(ks, cc, ls='none', marker=MARK[t], ms=2.6, mfc=TONE[t], mec='white', mew=0.35, zorder=4)
    g = GAM6.mean(); kx = np.arange(6, 80)
    ax.plot(kx, (DEGS['BA'] >= 6).mean() * ((kx - 0.5) / 5.5) ** (1 - g), color=INK, lw=0.7,
            ls=(0, (3.2, 2.2)), zorder=2)
    ax.text(24, 0.045, f'$\\hat{{\\gamma}}$ = {g:.2f}', fontsize=6.0, color='k', ha='left', va='bottom')
    ax.set_xlabel('degree  $k$', labelpad=1.6); ax.set_ylabel(r'$P(\geq k)$', labelpad=2.0)
    heading(fig, ox, oy, PW_B, PH_B, letter, 'Degree distribution')

def draw_cascade(fig, ox, oy, letter):
    ax = axes_at(fig, ox, oy, BOX_R if MIRROR[letter] else BOX_L)
    frame(ax, [0, 10, 20], [0, 3, 6], right=MIRROR[letter])
    tt = np.arange(21)
    for t in TOPOS:
        r = RATIO[t][:, :21]; m, sd = r.mean(0), r.std(0)
        ax.fill_between(tt, m - sd, m + sd, color=TONE[t], alpha=0.15, lw=0, zorder=2)
        ax.plot(tt, m, color=TONE[t], lw=1.0 if t == 'BA' else 0.9, zorder=3)
        ax.plot([P02.TAU], [DHP[t].mean()], ls='none', marker=MARK[t], ms=3.4, mfc=TONE[t], mec='white',
                mew=0.4, zorder=5)
    guide(ax, P02.TAU, r' $\tau$ = 5', 0.86)
    ax.set_xlabel('step after the seed', labelpad=1.6)
    ax.set_ylabel('hub over peripheral', labelpad=2.0)
    ax.legend(handles=[Line2D([], [], color=TONE[t], lw=0.9, marker=MARK[t], ms=2.6, mfc=TONE[t], mec='white',
                              mew=0.35, label=LABEL[t]) for t in TOPOS],
              loc='upper right', fontsize=5.8, handlelength=1.5, handletextpad=0.4, labelspacing=0.2,
              borderaxespad=0.15, frameon=False)
    heading(fig, ox, oy, PW_B, PH_B, letter, 'Hub-first cascade')

def draw_removal(fig, ox, oy, letter):
    ax = axes_at(fig, ox, oy, BOX_R if MIRROR[letter] else BOX_L)
    frame(ax, [0, 0.25, 0.5], [0, 0.5, 1], right=MIRROR[letter])
    for t in TOPOS:
        kinds = [('rand', (0, (2.6, 1.6)))] if t == 'LAT' else [('rand', (0, (2.6, 1.6))), ('targ', '-')]
        for k, ls in kinds:
            v = KEEP[t][k]; m, sd = v.mean(0), v.std(0)
            if k == 'targ':
                ax.fill_between(Q, np.clip(m - sd, 0, 1), np.clip(m + sd, 0, 1), color=TONE[t],
                                alpha=0.15, lw=0, zorder=2)
            ax.plot(Q, m, color=TONE[t], ls=ls, lw=1.0 if t == 'BA' else 0.9, zorder=3)
            ax.plot(Q[1:], m[1:], ls='none', marker=MARK[t], ms=2.4, mfc=TONE[t], mec='white', mew=0.35, zorder=4)
    guide(ax, 0.28, '$q$ = 0.28 ', 0.86, ha='right')
    ax.legend(handles=[Line2D([], [], color=INK, lw=0.9, ls=(0, (2.6, 1.6)), label='at random'),
                       Line2D([], [], color=INK, lw=1.0, ls='-', label='hubs first')],
              loc='lower left', fontsize=6.0, handlelength=2.0, handletextpad=0.5, labelspacing=0.3,
              borderaxespad=0.2, frameon=True, fancybox=False, framealpha=1.0, facecolor='white',
              edgecolor='none', borderpad=0.25)
    ax.set_xlabel('fraction removed  $q$', labelpad=1.6)
    ax.set_ylabel('largest component', labelpad=2.0)
    heading(fig, ox, oy, PW_B, PH_B, letter, 'Robust yet fragile')

TOP = [('A', 'LAT'), ('B', 'WS'), ('C', 'ER'), ('D', 'BA')]
BOTTOM = [('E', draw_cascade), ('F', draw_degree), ('G', draw_removal)]

for letter, t in TOP:
    fig = plt.figure(figsize=(PW_T, PH_T)); fig.set_label('Manuscript Fig 2' + letter)
    draw_structure(fig, 0.0, 0.0, t, letter)
    plt.show()
for letter, draw in BOTTOM:
    fig = plt.figure(figsize=(PW_B, PH_B)); fig.set_label('Manuscript Fig 2' + letter)
    draw(fig, 0.0, 0.0, letter)
    plt.show()

b = np.concatenate([np.log2(SHOW_W[t] * len(SHOW_W[t])) for t in TOPOS])
print(f'drawn structures, largest components: '
      + ', '.join(f'{LABEL[t]} {SHOW[t].number_of_nodes()} elements' for t in TOPOS))
print(f'log2(p_i n) of the drawn elements spans {b.min():+.2f} to {b.max():+.2f} bits; '
      f'{int((np.abs(b) > VLIM).sum())} of {b.size} lie beyond the scale of the second paper, ±{VLIM:.2f}, '
      f'and are drawn at its end')

drawn structures, largest components: Regular lattice 100 elements, Small-world 100 elements, Random 100 elements, Scale-free 100 elements
log2(p_i n) of the drawn elements spans -2.97 to +2.07 bits; 2 of 400 lie beyond the scale of the second paper, ±2.72, and are drawn at its end


## Step 5. The figure set together

In [10]:
fig = plt.figure(figsize=(FIG_W, PH_T + PH_B)); fig.set_label('Manuscript Fig 2')
for i, (letter, t) in enumerate(TOP):
    draw_structure(fig, i * PW_T, PH_B, t, letter)
for i, (letter, draw) in enumerate(BOTTOM):
    draw(fig, i * PW_B, 0.0, letter)
bold = sorted(x.get_text() for x in fig.findobj(mpl.text.Text) if x.get_fontweight() in ('bold', 700) and x.get_text().strip())
print('bold type only in the panel letters:', bold == list('ABCDEFG'))
plt.show()

bold type only in the panel letters: True
